# DAY 1 - LLM Learning

## LLM Foundations

### What I Will Learn

Today I will learn the basic foundations of Large Language Models (LLMs):

1. What is an LLM?
2. Tokens
3. Embeddings
4. Transformer intuition
5. Context windows
6. Temperature
7. Top-p
8. LLM hallucinations

The goal is to understand how LLMs process text and how different generation settings affect their responses.

# 1. What is an LLM?

A Large Language Model (LLM) is a machine learning model designed to understand and generate human-like text.

An LLM learns patterns from a very large amount of training data.

At a high level, an LLM:

- receives text as input
- converts the text into tokens
- processes the tokens using neural network layers
- uses learned patterns to predict likely next tokens
- generates an output sequence

A simple view of the process is:

Text → Tokens → Neural Network → Predicted Tokens → Text

Examples of tasks that LLMs can perform:

- Question answering
- Text generation
- Summarization
- Translation
- Classification
- Code generation

In [10]:


text = "Large Language Models can generate human-like text."

print("Input text:")                                          
print(text)

Input text:
Large Language Models can generate human-like text.


# 2. Tokens

LLMs do not directly process raw sentences as humans do.

Text is first broken into smaller units called **tokens**.

A token can be:

- a complete word
- part of a word
- punctuation
- a special symbol

For example, a sentence such as:

"Hello, how are you?"

may be divided into multiple tokens.

The exact tokenization depends on the tokenizer used by the model.

### Why are tokens important?

Tokens are important because:

- LLMs process text in token form.
- Context windows are measured in tokens.
- API usage and cost are often related to the number of tokens.
- Different languages and types of text can tokenize differently.

## Tokenization Experiment

The onboarding curriculum recommends observing how different types of text tokenize differently, including English, code, and Hindi.

For this experiment, we can use the `tiktoken` library to inspect token IDs.

> Note: Tokenization is model/tokenizer-specific. The tokenizer below is used only as a practical demonstration.

In [11]:
# Install tiktoken if it is not already installed.
# Run this cell only if the import in the next cell fails.

%pip install tiktoken

Note: you may need to restart the kernel to use updated packages.


In [12]:
import tiktoken

# Use a commonly available OpenAI tokenizer encoding
encoding = tiktoken.get_encoding("cl100k_base")

texts = [
    "Hello, how are you?",
    "print('Hello World')",
    "नमस्ते, आप कैसे हैं?"
]

for text in texts:
    tokens = encoding.encode(text)

    print("Text:", text)
    print("Token IDs:", tokens)
    print("Number of tokens:", len(tokens))
    print("-" * 50)

Text: Hello, how are you?
Token IDs: [9906, 11, 1268, 527, 499, 30]
Number of tokens: 6
--------------------------------------------------
Text: print('Hello World')
Token IDs: [1374, 493, 9906, 4435, 873]
Number of tokens: 5
--------------------------------------------------
Text: नमस्ते, आप कैसे हैं?
Token IDs: [61196, 88344, 79468, 31584, 97, 35470, 11, 15272, 228, 87262, 48909, 12906, 230, 79468, 35470, 85410, 12906, 230, 73414, 30]
Number of tokens: 20
--------------------------------------------------


# 3. Embeddings

An embedding is a numerical representation of information such as text.

Instead of representing a sentence only as characters or words, an embedding represents its meaning or characteristics as a vector of numbers.

For example:

"king" → [numerical vector]

"queen" → [numerical vector]

Similar concepts can have vectors that are closer together in an embedding space.

### Embeddings are useful for:

- Semantic search
- Similarity comparison
- Retrieval-Augmented Generation (RAG)
- Recommendation systems
- Clustering

### Tokens vs Embeddings

**Tokens** represent pieces of text.

**Embeddings** represent information as numerical vectors.

In [13]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [14]:
import numpy as np

# A simple toy example of vectors.
# These are NOT real LLM embeddings.
# They are only used to understand the idea of vectors.

cat = np.array([0.9, 0.8, 0.2])
dog = np.array([0.85, 0.75, 0.25])
car = np.array([0.1, 0.2, 0.9])

print("Cat vector:", cat)
print("Dog vector:", dog)
print("Car vector:", car)

Cat vector: [0.9 0.8 0.2]
Dog vector: [0.85 0.75 0.25]
Car vector: [0.1 0.2 0.9]


# 4. Transformer Intuition

Transformers are a key architecture behind modern LLMs.

One of the most important ideas in a Transformer is **attention**.

Attention allows the model to consider relationships between different tokens in the input.

For example:

"The animal didn't cross the road because it was tired."

To understand what "it" refers to, the model needs to consider the relationship between words in the sentence.

### Simplified Transformer flow

Input text
↓
Tokenization
↓
Token representations
↓
Attention
↓
Neural network layers
↓
Output representations
↓
Next-token prediction

The attention mechanism helps the model determine which parts of the input are important when processing a token.

## Scaled Dot-Product Attention

A simplified attention calculation uses:

Attention(Q, K, V) = softmax(QKᵀ / √dₖ)V

Where:

- Q = Query
- K = Key
- V = Value
- dₖ = dimension of the key vectors

The exact Transformer architecture contains many additional components, but this calculation helps demonstrate the basic intuition behind attention.

In [15]:
import numpy as np

def softmax(x):
    exp_x = np.exp(x - np.max(x))
    return exp_x / exp_x.sum(axis=-1, keepdims=True)

# Small toy Query, Key and Value matrices
Q = np.array([
    [1.0, 0.0],
    [0.0, 1.0]
])

K = np.array([
    [1.0, 0.0],
    [0.0, 1.0]
])

V = np.array([
    [10.0, 0.0],
    [0.0, 20.0]
])

d_k = K.shape[1]

# Calculate attention scores
scores = Q @ K.T / np.sqrt(d_k)

# Convert scores to probabilities
attention_weights = softmax(scores)

# Calculate weighted values
attention_output = attention_weights @ V

print("Attention scores:")
print(scores)

print("\nAttention weights:")
print(attention_weights)

print("\nAttention output:")
print(attention_output)

Attention scores:
[[0.70710678 0.        ]
 [0.         0.70710678]]

Attention weights:
[[0.66976155 0.33023845]
 [0.33023845 0.66976155]]

Attention output:
[[ 6.69761549  6.60476901]
 [ 3.30238451 13.39523099]]


# 5. Context Window

A context window is the amount of information, measured in tokens, that a model can consider within a particular request/conversation context.

For example, if a model has a context limit of N tokens, the input and other context must fit within the available token limit.

### Why is the context window important?

A larger context window can allow a model to work with:

- Longer documents
- Longer conversations
- More instructions
- More examples
- More retrieved information

However, a larger context window does not automatically mean the model will use every piece of information perfectly.

### Important

Context windows are model-specific and can change as models are updated.

In [16]:
# Simple token counting experiment

text = """
Large Language Models process text using tokens.
Tokens are smaller units of text used by the model.
"""

tokens = encoding.encode(text)

print("Text:")
print(text)

print("Number of tokens:", len(tokens))

Text:

Large Language Models process text using tokens.
Tokens are smaller units of text used by the model.

Number of tokens: 20


### Observation

The number of tokens in the input contributes to the amount of context used by a model.

This is why token counting is useful when working with long documents, APIs, and RAG systems.

# 6. Temperature

Temperature is a parameter used during text generation to influence the randomness of token selection.

A simplified intuition:

- Lower temperature → more predictable and consistent output
- Higher temperature → more varied and random output

Temperature does not add new knowledge to a model.

It changes how probabilities are used when selecting the next token.

### Example

A model might assign probabilities:

apple = 0.60
banana = 0.25
orange = 0.10
grape = 0.05

A lower temperature makes the highest-probability choices more dominant.

A higher temperature makes lower-probability choices more likely to be selected.

# 7. Top-p

Top-p is another sampling parameter used during text generation.

Instead of considering every possible token, top-p sampling selects from the smallest group of tokens whose cumulative probability reaches the chosen threshold.

For example, if:

A = 0.50
B = 0.25
C = 0.15
D = 0.10

With top-p = 0.75:

A + B = 0.75

So the sampling pool can contain A and B.

### Simple intuition

Lower top-p:
- Smaller candidate pool
- More focused generation

Higher top-p:
- Larger candidate pool
- More possible choices

Temperature and top-p affect sampling differently and can be used to control generation behavior.

# 4. Day 1 Summary

Today I learned the basic concepts of Large Language Models (LLMs).

### Topics Covered

- What is an LLM?
- How LLMs process text
- What are tokens?
- Tokenization using `tiktoken`
- What are embeddings?
- Difference between tokens and embeddings
- Vector representation
- Cosine similarity
- Basic Python experiments with vectors

### Key Takeaways

- LLMs process text as tokens.
- Tokens are smaller units of text.
- Embeddings represent information as numerical vectors.
- Similar vectors can indicate similar representations.
- Cosine similarity can be used to compare vectors.
- Python libraries such as `tiktoken` and `numpy` can be used for these experiments.